In [2]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [4]:
# 데이터 준비 및 TF-IDF 벡터화

from sklearn.feature_extraction.text import TfidfVectorizer


# 전처리된 텍스트 사용
texts = document_df['processed_text']

# TF-IDF 벡터화
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(texts)

# PyCaret 입력용 DataFrame 변환
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

In [5]:
# 2. PyCaret 군집화 환경 설정

from pycaret.clustering import setup, create_model, assign_model, models
from sklearn.metrics import silhouette_score

# PyCaret 세팅 (silent 제거, normalize 옵션 유지)
clu = setup(tfidf_df, session_id=123, normalize=True)

# 사용 가능한 군집화 모델 확인
print(models())

,Description,Value
0,Session id,123
1,Original data shape,"(51, 5000)"
2,Transformed data shape,"(51, 5000)"
3,Numeric features,5000
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore


                                       Name  \
ID                                            
kmeans                   K-Means Clustering   
ap                     Affinity Propagation   
meanshift             Mean Shift Clustering   
sc                      Spectral Clustering   
hclust             Agglomerative Clustering   
dbscan     Density-Based Spatial Clustering   
optics                    OPTICS Clustering   
birch                      Birch Clustering   
kmodes                   K-Modes Clustering   

                                                   Reference  
ID                                                            
kmeans                        sklearn.cluster._kmeans.KMeans  
ap         sklearn.cluster._affinity_propagation.Affinity...  
meanshift              sklearn.cluster._mean_shift.MeanShift  
sc              sklearn.cluster._spectral.SpectralClustering  
hclust     sklearn.cluster._agglomerative.AgglomerativeCl...  
dbscan                        sklearn.clu

In [7]:
from pycaret.clustering import setup, create_model, assign_model
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def evaluate_clustering_models(tfidf_df, models_list=None, session_id=123):
    """
    여러 군집화 모델을 자동 실행하고 라벨 개수에 따라 적절한 지표로 평가하는 함수
    
    Parameters:
        tfidf_df (pd.DataFrame): TF-IDF 벡터화된 데이터프레임
        models_list (list): 평가할 모델 리스트 (기본값: ['kmeans','dbscan','hdbscan','agglomerative'])
        session_id (int): PyCaret 세션 ID
    
    Returns:
        dict: 모델별 평가 점수
    """
    if models_list is None:
        models_list = ['kmeans','dbscan','hdbscan','agglomerative']
    
    # PyCaret 환경 설정
    setup(tfidf_df, session_id=session_id, normalize=True)
    
    results = {}
    
    for model_name in models_list:
        try:
            model = create_model(model_name)
            clustered_df = assign_model(model)
            labels = clustered_df['Cluster']
            
            # 라벨 개수 확인
            if len(set(labels)) > 1:
                # Silhouette Score
                score = silhouette_score(tfidf_df, labels)
                metric = "Silhouette"
            else:
                # 다른 지표 사용
                score = davies_bouldin_score(tfidf_df, labels)
                metric = "Davies-Bouldin"
            
            results[model_name] = {"metric": metric, "score": score}
        
        except Exception as e:
            results[model_name] = {"error": str(e)}
    
    return results

In [ ]:
scores = evaluate_clustering_models(tfidf_df)
print(scores)

results = '''
{
    'kmeans': {'metric': 'Silhouette', 'score': -0.05001975404267599}, 
    'dbscan': {'error': 'Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)'}, 
    'hdbscan': {'error': 'Estimator hdbscan not available. Please see docstring for list of available estimators.'}, 
    'agglomerative': {'error': 'Estimator agglomerative not available. Please see docstring for list of available estimators.'}
}
'''

# 정리
'''
    - 현재 PyCaret 3.x에서 기본 제공되는 군집화 모델은 KMeans, DBSCAN 정도라서 성능이 제한적
    - 차원 축소 + KMeans 또는 DBSCAN 파라미터 튜닝을 병행하면 더 나은 결과를 얻을 수 있음
    - HDBSCAN, Agglomerative 같은 고급 군집화는 PyCaret이 아닌 scikit-learn / hdbscan 라이브러리를 직접 써야 함.
'''

,Description,Value
0,Session id,123
1,Original data shape,"(51, 5000)"
2,Transformed data shape,"(51, 5000)"
3,Numeric features,5000
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,-0.0369,1.1096,0.9635,0,0,0


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0,0,0,0,0,0


{'kmeans': {'metric': 'Silhouette', 'score': -0.05001975404267599}, 'dbscan': {'error': 'Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)'}, 'hdbscan': {'error': 'Estimator hdbscan not available. Please see docstring for list of available estimators.'}, 'agglomerative': {'error': 'Estimator agglomerative not available. Please see docstring for list of available estimators.'}}
